In [ ]:
# notebooks/04_complete_portfolio.ipynb
"""
Complete Task 4: Portfolio Optimization with MPT
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("TASK 4: PORTFOLIO OPTIMIZATION")
print("="*60)

def load_returns(tickers):
    """Load and calculate returns with validation."""
    data = {}
    returns = pd.DataFrame()
    
    for ticker in tickers:
        try:
            df = pd.read_csv(f'../data/processed/{ticker}_data.csv', parse_dates=['Date'])
            df.set_index('Date', inplace=True)
            
            # Validate
            if df.empty:
                raise ValueError(f"Empty data for {ticker}")
            if 'Close' not in df.columns:
                raise ValueError(f"Missing 'Close' for {ticker}")
            
            data[ticker] = df
            returns[ticker] = df['Close'].pct_change()
            
        except Exception as e:
            print(f"Error loading {ticker}: {e}")
            raise
    
    returns.dropna(inplace=True)
    
    if returns.empty:
        raise ValueError("No valid returns data")
    
    print(f"✓ Loaded returns: {len(returns)} rows")
    return returns, data

# Load data
tickers = ['TSLA', 'BND', 'SPY']
returns, data = load_returns(tickers)

# Calculate historical returns
historical_returns = returns.mean() * 252
print(f"\nHistorical Returns (Annualized):")
for ticker in tickers:
    print(f"  {ticker}: {historical_returns[ticker]*100:.2f}%")

# Get forecasted TSLA return from Task 3
tsla_forecast_return = 0.15  # 15% annualized from Task 3

# Construct expected returns
expected_returns = pd.Series({
    'TSLA': tsla_forecast_return,
    'BND': historical_returns['BND'],
    'SPY': historical_returns['SPY']
})

# Covariance matrix
cov_matrix = returns.cov() * 252

print("\nExpected Returns (Combined):")
for ticker in expected_returns.index:
    source = "Forecasted" if ticker == 'TSLA' else "Historical"
    print(f"  {ticker}: {expected_returns[ticker]*100:.2f}% ({source})")

# Portfolio optimization
def portfolio_performance(weights):
    """Calculate portfolio performance with validation."""
    weights = np.array(weights)
    
    # Validate weights
    if abs(weights.sum() - 1) > 1e-6:
        raise ValueError(f"Weights don't sum to 1: {weights.sum()}")
    if np.any(weights < 0):
        raise ValueError(f"Negative weights: {weights}")
    
    ret = np.sum(expected_returns * weights)
    vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe = ret / vol if vol > 0 else 0
    
    return ret, vol, sharpe

# Find optimal portfolios
def optimize_portfolio(target_return=None):
    """Find optimal portfolio weights."""
    n_assets = len(tickers)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(n_assets))
    
    # Max Sharpe
    def neg_sharpe(weights):
        _, _, sharpe = portfolio_performance(weights)
        return -sharpe
    
    result_sharpe = minimize(neg_sharpe, 
                            n_assets * [1./n_assets],
                            method='SLSQP',
                            bounds=bounds,
                            constraints=constraints)
    
    # Min Volatility
    def volatility(weights):
        _, vol, _ = portfolio_performance(weights)
        return vol
    
    result_vol = minimize(volatility,
                         n_assets * [1./n_assets],
                         method='SLSQP',
                         bounds=bounds,
                         constraints=constraints)
    
    return result_sharpe.x, result_vol.x

try:
    max_sharpe_weights, min_vol_weights = optimize_portfolio()
    
    # Calculate performance
    max_sharpe_ret, max_sharpe_vol, max_sharpe_sharpe = portfolio_performance(max_sharpe_weights)
    min_vol_ret, min_vol_vol, min_vol_sharpe = portfolio_performance(min_vol_weights)
    
    print("\n" + "="*60)
    print("OPTIMAL PORTFOLIOS")
    print("="*60)
    
    print("\nMaximum Sharpe Ratio Portfolio:")
    print(f"  Return: {max_sharpe_ret*100:.2f}%")
    print(f"  Volatility: {max_sharpe_vol*100:.2f}%")
    print(f"  Sharpe Ratio: {max_sharpe_sharpe:.4f}")
    print("  Weights:")
    for i, ticker in enumerate(tickers):
        print(f"    {ticker}: {max_sharpe_weights[i]*100:.1f}%")
    
    print("\nMinimum Volatility Portfolio:")
    print(f"  Return: {min_vol_ret*100:.2f}%")
    print(f"  Volatility: {min_vol_vol*100:.2f}%")
    print(f"  Sharpe Ratio: {min_vol_sharpe:.4f}")
    print("  Weights:")
    for i, ticker in enumerate(tickers):
        print(f"    {ticker}: {min_vol_weights[i]*100:.1f}%")
    
    # Recommended portfolio
    print("\n" + "="*60)
    print("RECOMMENDED PORTFOLIO: Maximum Sharpe Ratio")
    print("="*60)
    print(f"\nWeights:")
    for i, ticker in enumerate(tickers):
        print(f"  {ticker}: {max_sharpe_weights[i]*100:.1f}%")
    print(f"\nExpected Annual Return: {max_sharpe_ret*100:.2f}%")
    print(f"Expected Volatility: {max_sharpe_vol*100:.2f}%")
    print(f"Sharpe Ratio: {max_sharpe_sharpe:.4f}")
    
except Exception as e:
    print(f"Optimization failed: {e}")
    raise

print("\n" + "="*60)
print("TASK 4 COMPLETE")
print("="*60)